# Multiprocessing

Or writing programs to use multiple processor cores

### This lesson is based on notes from [sebastianraschka.com](http://sebastianraschka.com/Articles/2014_multiprocessing.html)

What actually happens when you run code, or execute a program on a computer? If we focus on Python, what happens when we run Python code? Does our computer "speak Python", understand wh[...]


![](https://github.com/dustywhite7/Econ8320/blob/master/SlidesCode/serialParallel.png?raw=true)

One of the single biggest changes in computational technology in the recent past was the advent and spread of parallel computation. This concept is the crux of this lesson, but needs a lo[...]

One place where parallel processing really shines is in estimating complex mathematical calculations. While some math can be solved algebraically, other math problems can only be solved c[...]

In [ ]:
# define any function here!
def f(x):
    # return the value of the function given x
    return 1/(1 + x**2)

Now, we need to write a function that can sample from our function, and calculate the area of the sampled rectangles, and then return an approximate area.

In [ ]:
import random

def serial_integral(nSample, f, xmin, xmax):
  # determine points of estimation
  sample = sorted([random.uniform(xmin, xmax) for i in range(nSample)])
  # Calculate height at each point
  value = [f(i) for i in sample]
  # Calculate areas of rectangles
  # We have to specially calculate the first rectangle,
  #   because xmin is not part of our list of samples
  area = [(sample[0]-xmin)*value[0]] + [(sample[i]-sample[i-1])*value[i] for i in range(1, len(sample))]
  # Return sum as an approximate integral
  return sum(area)

This is our function for actually integrating a function `f` from `xmin` to `xmax` across `nSample` random intervals. Below is a picture of how this function gets (approximately) closer [...]

In [ ]:
import multiprocess as mp
# if needed, install with
# !pip install multiprocess

def serial_average(n_bins, n_reps, f, xmin, xmax):
  attempts = [serial_integral(n_bins, f, xmin, xmax) for i in range(n_reps)]
  return sum(attempts)/n_reps

def parallel_average(processes, n_bins, n_reps, f, xmin, xmax):
  pool = mp.Pool(processes=processes)
  results = [pool.apply_async(serial_integral, (n_bins, f, xmin, xmax)) for i in range(n_reps)]
  results = [p.get() for p in results]
  return sum(results)/n_reps

Now let's explore what that function does.

In [ ]:
def parallel_average(processes, n_bins, n_reps, f, xmin, xmax):
  pool = mp.Pool(processes=processes)
  ...
  return ...

The `mp.Pool` class provides the functionality to organize our processes, and to define the degree to which we want to spread our work across various **processes**. We can specify how ma[...]

In [ ]:
def parallel_average(processes, n_bins, n_reps, f, xmin, xmax):
  ...
  results = [pool.apply_async(serial_integral, (n_bins, f, xmin, xmax)) for i in range(n_reps)]
  ...
  return ...

We next use the `apply_async` method to pass the values that we want our pooled instances to calculate. We need to provide the function to be executed, as well as the arguments for the f[...]

In [ ]:
def parallel_average(processes, n_bins, n_reps, f, xmin, xmax):
  ...
  results = [p.get() for p in results]
  ...
  return ...

The next step is to use the `get()` method on each element of our returned processes. This will fetch the return statement values from each of the processes that we executed in the last [...]

In [ ]:
import time

benchmarks = [] # list to store our execution times

# Benchmark serial code
start = time.time()
result = serial_average(10000, 10000, f, 0, 1)
end = time.time()
elapsed = end - start
print(f'Serial function time taken: {elapsed:.6f} seconds') # library for timing execution of code

benchmarks.append(elapsed)

# Benchmark parallel code
start = time.time()
# Run a process for each CPU core minus one
result = parallel_average(mp.cpu_count()-1, 10000, 10000, f, 0, 1)
end = time.time()
elapsed = end - start
print(f'Parallel function time taken: {elapsed:.6f} seconds') # library for timing execution of code

benchmarks.append(elapsed)

Serial function time taken: 35.588530 seconds
Parallel function time taken: 4.581346 seconds


Amazing! When timing the output on my virtual machine (not at all a powerful computer), the parallel version runs nearly twice as fast (0.36 seconds vs 0.69 seconds). Even in a fairly tr[...]

## Solve-it!

Simulate 1,000 draws from a multivariate normal distribution and calculate the average value of the dependent variable, but do it 100 times! This is called "bootstrapping" and is a cri[...]

In [ ]:
#si-exercise
import numpy as np
import timeit
import multiprocessing as mp

def generate_y(seed=None):
    if seed is not None:
        np.random.seed(seed)

    n = 1000

    alpha = np.random.normal(15, 2, n)
    x1 = np.random.normal(3, 5, n)
    x2 = np.random.normal(10, 1, n)
    x3 = np.random.normal(8, 8, n)
    epsilon = np.random.normal(0, 1, n)

    y = alpha + x1 + 2*x2 + 0.5*x3 + epsilon

    return float(np.mean(y))

def sCalc(n=1000):
    return [generate_y(i) for i in range(n)]

def pCalc(n=1000):
    seeds = list(range(n))
    with mp.Pool(2) as pool:
        results = pool.map(generate_y, seeds)
    return results

sTime = timeit.Timer('sCalc(1000)', 'from __main__ import sCalc').timeit(number=100)

pTime = timeit.Timer('pCalc(1000)', 'from __main__ import pCalc').timeit(number=100)

sResults = sCalc(1000)
pResults = pCalc(1000)

print("Serial results:", sResults[:5])
print("Parallel results:", pResults[:5])
print("Serial time:", sTime)
print("Parallel time:", pTime)
print("Difference:", pTime - sTime)